In [3]:
from scipy.stats import norm
from scipy.special import erf
import numpy as np

In [18]:
"""The pricing function of European call option"""
def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float, K: Union[float, List[float], np.ndarray]) -> np.ndarray:
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    """
    # check conditions
    assert sigma > 0

    K = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)
    pdf_d1 = norm.pdf(d1_vec)

    price = N_d1_vec * S0 - K * np.exp((-1.0)*r*T) * N_d2_vec
    delta = N_d1_vec
    gamma = pdf_d1 / (S0 * sigma * np.sqrt(T))
    theta = -(S0 * pdf_d1 * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * N_d2_vec
    vega = S0 * np.sqrt(T) * pdf_d1
    rho = K * T * np.exp(-r * T) * N_d2_vec

    return {
        "price": price,
        "delta": delta,
        "gamma": gamma,
        "theta": theta,
        "vega": vega,
        "rho": rho
    }

In [16]:
black_scholes_eur_call(r=0.05, T=3, S0=20.0, sigma=0.3, K=30)

{'price': 2.326133193485558,
 'delta': 0.40833300470190736,
 'gamma': 0.03737034032018726,
 'theta': -0.9646924707910003,
 'vega': 13.453322515267413,
 'rho': 17.521580701657765}

In [19]:
"""The pricing function of European put option"""
def black_scholes_eur_put(r: float, T: float, S0: float, sigma: float, K: Union[float, List[float], np.ndarray]) -> np.ndarray:
    """
    Black-Scholes pricer of European put option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    """
    # check conditions
    assert sigma > 0

    K = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(-d1_vec)
    N_d2_vec = norm.cdf(-d2_vec)

    return K * np.exp((-1.0)*r*T) * N_d2_vec - N_d1_vec * S0

In [20]:
black_scholes_eur_put(r=0.05, T=3, S0=20.0, sigma=0.3, K=30)

8.147372486237291

In [22]:
K = 30
S0 = 20
sigma = 0.3
T = 3
r = 0.05

Forward = S0 - K * np.exp((-1.0)*r*T)
call_price = black_scholes_eur_call(r=r, T=T, S0=S0, sigma=sigma, K=K)["price"]
put_price = black_scholes_eur_put(r=r, T=T, S0=S0, sigma=sigma, K=K)
Forward - call_price + put_price

0.0